# Steam Reviews & Player Engagement

**Portfolio edition of the uploaded group analysis.** This notebook rebuilds the review sentiment and game-level regression workflow with portable file paths. It is a curated adaptation, not the complete original group submission. Original research also includes genre-specific comparisons and reviewer clustering. Dataset redistribution and licensing have not been verified.

## 1. Setup

Place `clean_review_proposal.csv` and `clean_steamspy_proposal.csv` in the `data/` directory. Run from this project folder. Dependencies are listed in `requirements.txt`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DATA_DIR = Path('data')
review_path = DATA_DIR / 'clean_review_proposal.csv'
steam_path = DATA_DIR / 'clean_steamspy_proposal.csv'
if not review_path.exists() or not steam_path.exists():
    raise FileNotFoundError('Add both source CSVs to data/ before running.')
reviews = pd.read_csv(review_path)
steam = pd.read_csv(steam_path)
print('Reviews:', reviews.shape, 'Games:', steam.shape)
print('Review columns:', reviews.columns.tolist())

## 2. Data preparation and validation

The unit of observation differs between review-level and game-level tables. Aggregate reviews to `appid` before merging to avoid duplicating game outcomes.

In [ ]:
reviews['review_date'] = pd.to_datetime(reviews['review_date'], errors='coerce')
for col in ['voted_up_int', 'review_length', 'author.playtime_forever']:
    reviews[col] = pd.to_numeric(reviews[col], errors='coerce')
for col in ['median_forever', 'ccu']:
    steam[col] = pd.to_numeric(steam[col], errors='coerce')
print(reviews[['appid','review_date','voted_up_int','review_length']].isna().mean().round(3))
print('Unique review game IDs:', reviews['appid'].nunique())
print('Unique game table IDs:', steam['appid'].nunique())

## 3. Sentiment scoring and feature engineering

VADER provides a compound polarity score for review text. It is a heuristic sentiment measure, not a direct measure of engagement.

In [ ]:
analyzer = SentimentIntensityAnalyzer()
reviews['vader_compound'] = reviews['review'].fillna('').astype(str).map(lambda s: analyzer.polarity_scores(s)['compound'])
reviews['is_short_review'] = (reviews['review_length'] <= 5).astype(int)
summary = reviews.groupby('appid').agg(
    positive_ratio=('voted_up_int', 'mean'),
    avg_vader_compound=('vader_compound', 'mean'),
    short_review_share=('is_short_review', 'mean'),
    review_count=('appid','size')
).reset_index()
assert summary['appid'].is_unique
print(summary.head())

## 4. Game-level regression

This portfolio edition fits an abbreviated model. The original group notebook also includes a `review_trend` feature; consequently, coefficients from this edition should **not** be presented as identical to those in the group report. HC3 standard errors address some forms of heteroskedasticity but do not establish causality.

In [ ]:
df = summary.merge(steam, on='appid', how='inner', validate='one_to_one')
df['log_median_forever'] = np.log1p(df['median_forever'].clip(lower=0))
df['log_ccu'] = np.log1p(df['ccu'].clip(lower=0))
features = ['log_median_forever','positive_ratio','avg_vader_compound','short_review_share','log_ccu']
model_df = df[features].replace([np.inf,-np.inf],np.nan).dropna()
print('Games in regression:', len(model_df))
model = smf.ols('log_median_forever ~ positive_ratio + avg_vader_compound + short_review_share + log_ccu', data=model_df).fit(cov_type='HC3')
print(model.summary())

## 5. Visualize results

Charts are generated from your local source data when you run the notebook. No fabricated screenshots or numerical outputs are included.

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(model_df['log_ccu'],model_df['log_median_forever'],alpha=0.65)
ax.set(xlabel='log(1 + peak concurrent users)',ylabel='log(1 + median playtime)',title='Popularity and player engagement')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

coefs = model.params.drop('Intercept')
errors = (model.conf_int().loc[coefs.index,1] - model.conf_int().loc[coefs.index,0])/2
fig, ax = plt.subplots(figsize=(8,4))
ax.errorbar(coefs.values,range(len(coefs)),xerr=errors.values,fmt='o',capsize=3)
ax.axvline(0,color='gray',linestyle='--')
ax.set_yticks(range(len(coefs)),coefs.index)
ax.set(xlabel='OLS coefficient with 95% confidence interval',title='Regression estimates (abbreviated specification)')
plt.tight_layout()
plt.show()

## 6. Interpretation and limitations

Compare coefficient directions and uncertainty intervals rather than interpreting correlations as causal effects. The original report's conclusions were based on a fuller specification and a finite sample of matched games. Potential selection bias, genre differences, measurement noise in review sentiment, and time-window mismatches remain important limitations.

**Further work:** Reproduce the full original review-trend feature, validate the genre-level analyses, and publish reviewer segmentation with appropriate clustering diagnostics and genuine figures.